In [1]:
%load_ext autoreload
%autoreload 2

In [189]:
import os

# CPU fallback (if GPU is not available)
if 'JAX_PLATFORMS' not in os.environ:
    try:
        from jax import devices
        if not any(d.platform == 'gpu' for d in devices()):
            os.environ['JAX_PLATFORMS'] = 'cpu'
    except Exception:
        os.environ['JAX_PLATFORMS'] = 'cpu'

# Import necessary libraries
import jax
import jax.numpy as jnp
import jax.random as random
import optax
from tqdm import tqdm
from utils import DataLoader
from flax import nnx
from flax.nnx.training.metrics import Metric, Average

# FIRST TRY: basic linear function

In [190]:
class MLP(nnx.Module):

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int, num_hidden_layers: int, *, rngs: nnx.Rngs):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.num_hidden_layers = num_hidden_layers
        if num_hidden_layers > 0:
            self.hidden_layers = [nnx.Linear(self.input_dim, self.hidden_dim, rngs=rngs)] + \
                [nnx.Linear(self.hidden_dim, self.hidden_dim, rngs=rngs) for _ in range(self.num_hidden_layers - 1)]
            self.linear = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        else:
            self.linear = nnx.Linear(self.input_dim, self.output_dim, rngs=rngs)

    def __call__(self, x: jax.Array):
        if self.num_hidden_layers > 0:
            for i in range(self.num_hidden_layers):
                x = self.hidden_layers[i](x)
                x = nnx.relu(x) 
            x = self.linear(x) 
        else:
            x = self.linear(x)
        return x

In [194]:
# DATASET GENERATION
key = random.key(0)
#f_to_learn = lambda mu, l, k, x: (mu+l+k)*x
f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(-l*x)

N = 1000
key, subkey = random.split(key) 
x = random.uniform(subkey, (N,), minval=-1, maxval=1)
key, subkey = random.split(key) 
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key) 
l = random.uniform(subkey, (N,), minval=0, maxval=1)
key, subkey = random.split(key) 
k = random.uniform(subkey, (N,), minval=-2, maxval=2)
y = f_to_learn(mu, l, k, x)
X = jnp.stack([mu, l, k, x], axis=1)

# DATASET SPLIT AND DATALOADER
split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
train_dataloader = DataLoader(X_train, y_train, batch_size=8, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=8, shuffle=False)

In [195]:
@nnx.jit
@nnx.vmap
def apply(network, parameters, x):
    """
    Assigns the parameters from an array to the state. state and parameters must be batched with the same first dimension.
    """
    parameters = parameters.squeeze()
    graphdef, state = nnx.split(network)
    flat_state = nnx.to_flat_state(state)

    # TODO: Check if the number of parameters matches
    
    i = 0
    for key, param in flat_state:
        param_size = param.value.size
        # Extract parameters for this specific parameter
        param_values = parameters[i:i + param_size]
        # Reshape to match the original parameter shape
        param.value = param_values.reshape(param.value.shape)
        i += param_size

    modified_network = nnx.merge(graphdef, state)

    return modified_network(x)


def train_step(hypernetwork, targetnetwork_fun, hyperparams, x, y, optimizer, batch_size):
    """
    Performs a single training step."""
    def loss_fn(hypernetwork, hyperparams, x, y, batch_size):
        w = hypernetwork(hyperparams)

        @nnx.split_rngs(splits=batch_size)
        @nnx.vmap(in_axes=(0, None), out_axes=0)
        def make_model(rngs, targetnetwork_fun):
            return targetnetwork_fun(1, 1, 8, 1, rngs=rngs)
        
        targetnetwork = make_model(nnx.Rngs(0), targetnetwork_fun)

        pred = apply(targetnetwork, w, x)
        loss = jnp.mean(optax.l2_loss(pred, y))
        return loss
    loss, grads = nnx.value_and_grad(loss_fn)(hypernetwork, hyperparams, x, y, batch_size)
    optimizer.update(grads)
    return loss

train_step = nnx.jit(train_step, static_argnames=('targetnetwork_fun','batch_size'))

def evaluation_step(hypernetwork, targetnetwork_fun, hyperparams, x, y, batch_size):
    """
    Performs a single training step."""
    def loss_fn(hypernetwork, hyperparams, x, y, batch_size):
        w = hypernetwork(hyperparams)

        @nnx.split_rngs(splits=batch_size)
        @nnx.vmap(in_axes=(0, None), out_axes=0)
        def make_model(rngs, targetnetwork_fun):
            return targetnetwork_fun(1, 1, 8, 1, rngs=rngs)
        
        targetnetwork = make_model(nnx.Rngs(0), targetnetwork_fun)
        
        pred = apply(targetnetwork, w, x)
        loss = jnp.mean(optax.l2_loss(pred, y))
        return loss
    loss = loss_fn(hypernetwork, hyperparams, x, y, batch_size)
    return loss

evaluation_step = nnx.jit(evaluation_step, static_argnames=('targetnetwork_fun','batch_size'))

In [196]:
# FOR NOW: TARGET NETWORK WITH KNOWN FIXED NUMBER OF PARAMETERS
num_params = 25
hypernetwork = MLP(input_dim=3, output_dim=num_params, hidden_dim=8, num_hidden_layers=1, rngs=nnx.Rngs(0))


optimizer = nnx.Optimizer(hypernetwork, optax.adam(learning_rate=1e-3))
training_loss = Average(argname='loss')
test_loss = Average(argname='loss')
epochs = 200
pbar = tqdm(range(epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}")
    training_loss.reset()
    test_loss.reset()
    for data, label in train_dataloader:
        hyperparams = data[:, :-1] # mu, l, k
        x = data[:, -1:] # x
        
        loss_step = train_step(hypernetwork, MLP, hyperparams, x, label, optimizer, x.shape[0])
        training_loss.update(loss = loss_step.item()) #TODO: CONTROLLARE SE FUNZIONA (IN PARTICOLARE SE TIENE CONTO CHE L'ULTIMO BATCH POTREEBBE ESSERE PIU' PICCOLO)
    for data, label in test_dataloader:
        hyperparams = data[:, :-1]
        x = data[:, -1:]
        loss_step = evaluation_step(hypernetwork, MLP, hyperparams, x, label, x.shape[0])
        test_loss.update(loss = loss_step.item())
    pbar.set_postfix({"training loss": training_loss.compute(), "test loss": test_loss.compute()})

Epoch 200: 100%|██████████| 200/200 [02:26<00:00,  1.36it/s, training loss=0.21564883, test loss=0.28704944]


# PROVA CODICE NUOVO

In [164]:
# Creazioine batch di modelli
batch_size = 4

@nnx.jit
@nnx.split_rngs(splits=batch_size)
@nnx.vmap
def make_model(rngs):
  return MLP(1, 1, 8, 0, rngs=rngs)

model = make_model(nnx.Rngs(0))

print(model)

MLP( # Param: 8 (32 B)
  hidden_dim=8,
  input_dim=1,
  linear=Linear( # Param: 8 (32 B)
    bias=Param( # 4 (16 B)
      value=Array(shape=(4, 1), dtype=dtype('float32'))
    ),
    bias_init=<function zeros at 0x7145dae3a7a0>,
    dot_general=<function dot_general at 0x7145db6eeca0>,
    dtype=None,
    in_features=1,
    kernel=Param( # 4 (16 B)
      value=Array(shape=(4, 1, 1), dtype=dtype('float32'))
    ),
    kernel_init=<function variance_scaling.<locals>.init at 0x714500745940>,
    out_features=1,
    param_dtype=float32,
    precision=None,
    promote_dtype=<function promote_dtype at 0x714500745bc0>,
    use_bias=True
  ),
  num_hidden_layers=0,
  output_dim=1
)


In [ ]:
# Provo a modificare parametri del batch di modelli con solita funzione

@nnx.vmap
def assign_parameters(state, parameters):
    """
    Assigns the parameters from an array to the state. state and parameters must be batched with the same first dimension.
    """
    parameters = parameters.squeeze()
    flat_state = nnx.to_flat_state(state)

    # TODO: Check if the number of parameters matches
    
    i=0
    for (key, param) in flat_state:
        param.value = parameters[i: i + param.value.size].reshape(param.value.shape)
        i += param.value.size   
    return state


graphdef, state = nnx.split(model)
w = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])

new_state = assign_parameters(state, w)

new_model = nnx.merge(graphdef, new_state)

@nnx.vmap(in_axes=(0, 0), out_axes=0)
def apply_batch(model, x):
    return model(x)

#new_model(jnp.array([[1], [2], [3], [4]]))  # NON SI PUO' USARE COSì, DEVO USARE APPLY_BATCH

y = apply_batch(new_model, jnp.array([[1], [2], [3], [4]]))  # Test the new model with some input
print(y)

# Nel codice sopra, applichiamo modello dentro funzione vmappata (funzione apply), quindi perfetto

[[ 3.]
 [11.]
 [23.]
 [39.]]


# BAD RESULT

This is wrong: every model receive the entire array [1,2,3,4] due to JAX's broadcasting rules.

In [ ]:
@nnx.jit
@nnx.split_rngs(splits=batch_size)
@nnx.vmap
def make_model(rngs):
  return MLP(1, 1, 8, 0, rngs=rngs)

models = make_model(nnx.Rngs(0))

models(jnp.array([1,2,3,4]))  # Test the new model with some input

Array([[61., 63., 65., 67.]], dtype=float32)

we need instead to do:

In [ ]:
# Reshape inputs to (4,1) for 4 models with input_dim=1
inputs = jnp.array([[1.0], [2.0], [3.0], [4.0]])

# Apply each input to corresponding model
@nnx.vmap(in_axes=(0, 0), out_axes=0)
def apply_individual(model, x):
    return model(x)

result = apply_individual(models, inputs)